# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jasleen13/ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/jasleen13/ML-Internship.git"
DATA_REL = Path("data") / "raw" / "content_refresh_anonymized.csv"

def find_repo_root(start: Path):
    p = start
    while not (p / DATA_REL).exists() and p != p.parent:
        p = p.parent
    return p if (p / DATA_REL).exists() else None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    clone_dir = Path("/content/ML-Internship") if Path("/content").exists() else Path.cwd() / "ML-Internship"
    if not (clone_dir / DATA_REL).exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    repo_root = clone_dir

DATA_PATH = repo_root / DATA_REL
df = pd.read_csv(DATA_PATH)
print("Using data at:", DATA_PATH)
print(df.shape)


Using data at: /content/ML-Internship/data/raw/content_refresh_anonymized.csv
(30000, 44)


## 1. Method choice and why

**The question shape:** "which pages first?", the same ranking question from Week 4, so
per the training-honest-models skill this calls for a classifier's probability evaluated at
precision@K, not a bare accuracy number.

**The label:** the same one the baseline already scores against, `is_underperforming`
(`ctr_gap < 0`, a page's CTR sits below its position tier's median). Same candidate pool as
Week 4: `impressions_90d >= 500`, `0 < avg_position <= 20`.

**A wrinkle worth stating up front, before any model gets fit:** `ctr_gap` is not just
correlated with the label, it IS the label, `is_underperforming` is a literal threshold on it.
Per the data contract's field-classification rule (label / proxy fields are never features) and
the leakage trap from notebook 02/03, `ctr`, `ctr_gap`, and `clicks_90d` are excluded from every
"honest" model below. That makes this genuinely harder than it looks: can content, position,
volume, and engagement signals predict which pages will underperform their tier's CTR,
*without ever looking at CTR itself*? That's a fair modeling question. It also means the Week 4
baseline is not a fair target to "beat", it built its score directly from `ctr_gap`, so it
achieves near-perfect precision@K by construction, not because it's a good model. Section 3
shows this explicitly rather than hiding it.

**Methods, in order of complexity:** Logistic Regression (readable, a sane floor above the dummy
classifier), Decision Tree (still readable, catches non-linear splits), Random Forest (usually
the strongest of the three, at the cost of readability). Permutation importance on whichever
wins, so the "what does it lean on" question in section 4 has a real answer.


In [2]:
visible = df["impressions_90d"] >= 500
in_range = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
lane = df[visible & in_range].copy()

tier_median_ctr = lane.groupby("position_tier")["ctr"].transform("median")
lane["ctr_gap"] = lane["ctr"] - tier_median_ctr
lane["is_underperforming"] = (lane["ctr_gap"] < 0).astype(int)

print(f"Candidate pool: {len(lane):,} pages (same as Week 4)")
print(f"Base rate (is_underperforming): {lane['is_underperforming'].mean():.3f}")
print(f"Distinct clients in pool: {lane['client_id'].nunique()}")


Candidate pool: 12,023 pages (same as Week 4)
Base rate (is_underperforming): 0.490
Distinct clients in pool: 28


## 2. Split design

**Client-grouped split (`GroupShuffleSplit` on `client_id`), 75/25.** Pages from the same
client share a template, an industry, and a content strategy, if the same client's pages land
in both train and test, the model can partly memorize "this is client X's style" rather than
learning something that generalizes to a client it has never seen. This is exactly the check the
lane guide's leakage checklist asks for: "are pages from the same client split across train and
test in a way that makes the test too easy?" The cell below confirms zero client overlap between
the two sides before any model gets fit.


In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_honest = ["impressions_90d", "avg_position", "word_count", "content_age_days",
                   "days_since_last_update", "engagement_rate", "scroll_rate", "ai_traffic_pct",
                   "search_volume", "competition", "cpc"]
categorical_honest = ["content_type", "main_intent", "position_tier"]

model_df = lane.copy()
for c in numeric_honest:
    model_df[c] = model_df[c].fillna(model_df[c].median())
for c in categorical_honest:
    model_df[c] = model_df[c].fillna("unknown")

X = model_df[numeric_honest + categorical_honest]
y = model_df["is_underperforming"]
groups = model_df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Train: {len(X_tr):,} rows, {groups.iloc[train_idx].nunique()} clients")
print(f"Test:  {len(X_te):,} rows, {groups.iloc[test_idx].nunique()} clients")
print(f"Client overlap between train and test: {len(overlap)} (must be 0)")

pre = ColumnTransformer([
    ("num", StandardScaler(), numeric_honest),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_honest),
])


Train: 11,199 rows, 21 clients
Test:  824 rows, 7 clients
Client overlap between train and test: 0 (must be 0)


## 3. Train + compare vs my baseline

Same split, same candidate pool, two metrics: ROC AUC and precision@K (K=20 and K=50, matching
the review-capacity framing from Week 4). Four honest rows (a dummy majority-class floor, then
the three real methods, none of them touching `ctr` or `ctr_gap`), plus one row that
deliberately re-adds `ctr_gap` to show what the Week 4 baseline is actually doing under the
hood, that row is not a model to be proud of, it's the leakage demonstration promised in
section 1.


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    top_k = np.asarray(y_true)[order[:k]]
    return top_k.mean()

rows = []
fitted = {}
for name, clf in [
    ("dummy_majority", DummyClassifier(strategy="most_frequent", random_state=42)),
    ("logistic_regression", LogisticRegression(max_iter=2000)),
    ("decision_tree", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ("random_forest", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
]:
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:, 1] if hasattr(pipe, "predict_proba") else pipe.predict(X_te).astype(float)
    auc = roc_auc_score(y_te, proba) if len(set(y_te)) > 1 else float("nan")
    rows.append({"model": name, "features": "honest (no ctr)", "roc_auc": auc,
                 "precision_at_20": precision_at_k(y_te.values, proba, 20),
                 "precision_at_50": precision_at_k(y_te.values, proba, 50)})
    fitted[name] = pipe

# The leakage demonstration: same logistic regression, ctr_gap re-added as a feature
numeric_leaky = numeric_honest + ["ctr_gap"]
X_leaky = model_df[numeric_leaky + categorical_honest]
X_tr_l, X_te_l = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]
pre_leaky = ColumnTransformer([
    ("num", StandardScaler(), numeric_leaky),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_honest),
])
pipe_leaky = Pipeline([("pre", pre_leaky), ("clf", LogisticRegression(max_iter=2000))])
pipe_leaky.fit(X_tr_l, y_tr)
proba_leaky = pipe_leaky.predict_proba(X_te_l)[:, 1]
rows.append({"model": "logistic_regression", "features": "LEAKY (ctr_gap included)",
             "roc_auc": roc_auc_score(y_te, proba_leaky),
             "precision_at_20": precision_at_k(y_te.values, proba_leaky, 20),
             "precision_at_50": precision_at_k(y_te.values, proba_leaky, 50)})

results = pd.DataFrame(rows)
print(f"Base rate in test set: {y_te.mean():.3f}")
results


Base rate in test set: 0.441


,model,features,roc_auc,precision_at_20,precision_at_50
0,dummy_majority,honest (no ctr),0.500000,0.50,0.42
1,logistic_regression,honest (no ctr),0.544558,0.75,0.64
2,decision_tree,honest (no ctr),0.663616,0.75,0.78
3,random_forest,honest (no ctr),0.726759,0.70,0.68
4,logistic_regression,LEAKY (ctr_gap included),1.000000,1.00,1.00


**Reading the table:** the dummy floor sits at the base rate, as it should, always guessing
the majority class tells you nothing about position or content. All three honest models beat
that floor, Random Forest wins on ROC AUC (0.727 vs 0.500 dummy), Decision Tree wins on
precision@50 (0.780), a reminder that "best on one metric" and "best on another" aren't always
the same model, worth reporting both rather than picking whichever flatters the winner.

The leaky row is the one to look at twice: ROC AUC 1.000, precision@20 and @50 both 1.000. That
is not a good model, it's `ctr_gap` recreating the exact threshold the label was built from,
the same shape as the notebook 02 and notebook 03 leakage demos. It's included here on purpose,
as direct evidence that the Week 4 baseline's near-perfect precision (it ranks by `ctr_gap`
directly) isn't a fair bar for these honest models to clear, they're answering a strictly harder
question by design. The real, honest comparison is Random Forest and Decision Tree against the
dummy floor, not against the baseline's `ctr_gap`-powered score.


## 4. Errors and interpretation

Permutation importance on the strongest honest model (Random Forest), then a look at where it's
actually wrong.


In [5]:
from sklearn.inspection import permutation_importance

rf_pipe = fitted["random_forest"]
perm = permutation_importance(rf_pipe, X_te, y_te, n_repeats=10, random_state=42, n_jobs=-1)
importances = pd.Series(perm.importances_mean, index=X_te.columns).sort_values(ascending=False)
print("Permutation importance, top 8:")
print(importances.head(8))


Permutation importance, top 8:
engagement_rate           0.157039
content_age_days          0.032039
word_count                0.014078
search_volume             0.011893
avg_position              0.007403
days_since_last_update    0.007403
scroll_rate               0.007282
impressions_90d           0.004005
dtype: float64


**What it leans on:** `engagement_rate` dominates every other feature by roughly 5x, that
makes sense rather than looking suspicious, pages where visitors don't stick around after
landing are plausibly the same pages that don't earn many clicks from the search results page
either, a related but genuinely distinct signal from CTR itself, not a backdoor to it.
`content_age_days` and `word_count` come in a distant second and third, older and thinner
content skewing toward underperformance is a believable, explainable pattern, not a "suspiciously
perfect" one per the training-honest-models skill's own sanity check.


In [6]:
proba = rf_pipe.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)
errors_df = model_df.iloc[test_idx].copy()
errors_df["pred_proba"] = proba
errors_df["pred"] = pred
errors_df["actual"] = y_te.values

false_neg = errors_df[(errors_df["actual"] == 1) & (errors_df["pred"] == 0)].sort_values("pred_proba")
false_pos = errors_df[(errors_df["actual"] == 0) & (errors_df["pred"] == 1)].sort_values("pred_proba", ascending=False)

print(f"False negatives (missed real underperformers): {len(false_neg)}")
print(false_neg[["content_id", "content_type", "position_tier", "impressions_90d",
                  "avg_position", "ctr", "pred_proba"]].head(3).to_string(index=False))
print()
print(f"False positives (flagged fine pages): {len(false_pos)}")
print(false_pos[["content_id", "content_type", "position_tier", "impressions_90d",
                  "avg_position", "ctr", "pred_proba"]].head(3).to_string(index=False))


False negatives (missed real underperformers): 82
          content_id    content_type position_tier  impressions_90d  avg_position  ctr  pred_proba
content_0d0461dca9e5 keyword article      striking             3705          15.9 0.16    0.253774
content_e8658537c97f keyword article      striking             8106          13.4 0.12    0.325284
content_2f06371514c2 keyword article      striking            25999          16.2 0.09    0.334940

False positives (flagged fine pages): 174
          content_id    content_type position_tier  impressions_90d  avg_position  ctr  pred_proba
content_f94f6a2b8453 keyword article      striking              594          18.1 0.17    0.751407
content_2bf26f7afc87 keyword article        page_1             4801           7.6 0.40    0.751288
content_0ef7040d504a keyword article      striking             2235          11.9 0.18    0.726290


**Three concrete wrong cases:**

1. **A false negative** (model said fine, actually underperforming): a `striking`-tier page with
   26,000 impressions and 0.09% CTR, well below its tier's median. The model's `pred_proba` sits
   around 0.33, low but not confidently wrong, this reads like a case where content/engagement
   signals alone genuinely don't carry enough information, the CTR problem here may be
   title-and-snippet-specific in a way none of the honest features capture.

2. **A false positive** (model flagged, actually fine): a `page_1` page with a strong 0.40% CTR
   that the model still scored at 0.75 probability of underperforming. Worth checking whether
   its `engagement_rate` happens to be unusually low for a page that's otherwise doing well,
   since that feature dominates the model, a mismatch there is the most likely explanation.

3. **A second false positive**: a low-volume `striking`-tier page (594 impressions) the model is
   fairly confident about (0.75) despite a CTR that isn't actually bad for its tier. Low-volume
   pages are exactly where a single feature (like `engagement_rate`) can look extreme just from
   small-sample noise, the same lesson Discovery B taught back in Week 1 about `feedly article`.

**What this says overall:** the honest model finds a real, explainable signal, engagement
quality plausibly relates to click-through quality, but it's a meaningfully weaker signal than
CTR itself, which is expected and not a failure. It's a genuinely harder question than the
baseline answers, and the errors above show where that difficulty actually lives: thin-volume
pages and pages where engagement and CTR diverge for reasons the current feature set can't see.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.